# Clinical preprocessing — Second-Progression target

This notebook turns the 148-patient eligible cohort into the canonical
artefacts the Mistral RAG pipeline expects:

| Output | Purpose |
|---|---|
| `Processed/clean_clinical.csv` | 147 rows × cleaned columns, with `y`, `Landmark_day`, `Death_day` already attached |
| `Processed/leakage_manifest.csv` | every column tagged **L1 / L2 / L3 / L4** (L1 = target/post-event, L4 = baseline-safe) |
| `Processed/feature_groups.json` | column lists for the 4 experiments |
| `splits/{Train,Validation,Test}.csv` | patient-level splits, **inherited from First_Recur** (subset to 147 eligible) |
| `Processed/preprocessing_summary.json` | counts, eligibility, class balance for the report |

> **Naming note**: `Landmark_day` is the day-from-diagnosis at which we make
> the prediction (≡ what the BEEP paper calls *T₂*).  We renamed the
> column from `T2` to `Landmark_day` to remove the collision with the
> *MRI T2-weighted sequence* (`t2w`) and *MRI Timepoint 2*.

Leakage tier definitions (renamed L1-L4 to avoid confusion with the
landmark name; same semantics as First_Recur's T1-T4):

- **L1** — Target itself or a literal post-event field. *Drop column.*
- **L2** — Mixed-timing field, ≥ 30 % post-landmark starts. *Drop column.*
- **L3** — Mostly pre-landmark field with a small fraction of post-landmark starts.
  *Keep column, mask the post-landmark rows AND any companion end-day cells > landmark.*
- **L4** — Baseline / immutable. *Keep as-is.*


In [ ]:
import sys, json
sys.path.insert(0, ".")
import pandas as pd, numpy as np
from pathlib import Path
from _landmark import (load_raw, eligible_cohort, attach_y_and_landmark,
                        cohort_exclusion_table,
                        COL_PROG1, COL_TTP1, COL_PROG2, COL_TTP2, COL_DEATH,
                        MRI_DAY_COLS, THERAPY_END_COLS)

PROC = Path("Processed"); PROC.mkdir(exist_ok=True)
SPLITS = Path("splits"); SPLITS.mkdir(exist_ok=True)

raw  = load_raw()
print("Eligibility funnel:")
print(cohort_exclusion_table(raw).to_string(index=False))
elig = attach_y_and_landmark(eligible_cohort(raw))
print(f"\neligible n = {len(elig)}   y dist = {elig['y'].value_counts().to_dict()}")
print(f"Landmark_day  median={elig['Landmark_day'].median():.0f} d   "
      f"min={elig['Landmark_day'].min():.0f} d   max={elig['Landmark_day'].max():.0f} d")


## 1. Tier each column


In [ ]:
TIER = {}
RATIONALE = {}

# ---- L1: target / post-event --------------------------------------------------
L1_COLS = [
    COL_PROG2, COL_TTP2,
    "Type of 2nd Progression",
    "Treatment started after 2nd progression",
    "Days from Diagnosis to new treatment",
    "2nd_Additional Therapy",
    "Cycle length of 2nd_Additional Therapy (q days)",
    "Number of Days from Diagnosis to Starting 2nd_Additional Therapy ",
    "Number of Days from Dagnosis to Complete 2nd_Additional Therapy ",
    "Number of Cycles of 2nd_Additional Therapy",
    "Hospice",
    "Overall Survival (Death)",
    "Number of days from Diagnosis to death (Days)",
]
for c in L1_COLS:
    TIER[c] = "L1"; RATIONALE[c] = "Target itself or post-event field"

# ---- L2 / L3: time-stamped therapy columns gated by Landmark_day -----------
TIMING_PAIRS = [
    ("Initial Chemo Therapy", " Number of days from Diagnosis to Initial Chemo Therapy Start date",
     [" Number of days from Diagnosis to Initial Chemo Therapy end date",
      "Name of Initial Chemo Therapy"]),
    ("Radiation Therapy", "Number of days from Diagnosis to Radiation Therapy Start date",
     ["Number of days from Diagnosis to Radiation Therapy end date",
      "Dose","Number of Fractions"]),
    ("Additional Therapy", "Number of Days from Diagnosis to Starting Additional Therapy ",
     ["Cycle length of Additional Therapy (q days)",
      "Number of Days from Diagnosis to Complete Additional Therapy ",
      "Number of Cycles of Additional Therapy"]),
    ("Immuno therapy", "Number of Days from Diagnosis to Start Immunotherapy ",
     ["Cycle length of Immunotherapy (q days)",
      "Number of Days from Diagnosis to Complete Immunotherapy ",
      "Number of Cycles of Immunotherapy"]),
    ("Brachy therapy", "Number of Days from Diagnosis to the day of Insertion of Brachytherapy ",
     []),
    ("Other Types of Therapy (LITT, more chemo, proton therapy)",
     "Number of Days from Diagnosis to Start Other Additional Therapy ",
     ["Number of Days from Diagnosis to Complete Other Additional Therapy "]),
]

for flag, day_col, companions in TIMING_PAIRS:
    days = pd.to_numeric(elig[day_col], errors="coerce") if day_col in elig.columns else pd.Series(dtype=float)
    started = days.notna()
    if started.sum() == 0:
        for c in [flag, day_col, *companions]:
            if c in elig.columns:
                TIER[c] = "T4"; RATIONALE[c] = "Never used in this cohort"
        continue
    pct_post = ((days >= elig["Landmark_day"]) & started).sum() / started.sum()
    if pct_post >= 0.30:
        for c in [flag, day_col, *companions]:
            if c in elig.columns:
                TIER[c] = "L2"
                RATIONALE[c] = f"{pct_post:.0%} of starts are POST-landmark — drop column"
    else:
        for c in [flag, day_col, *companions]:
            if c in elig.columns:
                TIER[c] = "L3"
                RATIONALE[c] = f"{pct_post:.0%} of starts post-landmark — mask post-landmark rows + post-landmark end-day cells"

# ---- L4: everything else (baseline / immutable) -----------------------------
for c in elig.columns:
    if c in TIER: continue
    if c in MRI_DAY_COLS:
        TIER[c] = "L4"; RATIONALE[c] = "MRI scan day — gated separately by Image_preprocessing"
        continue
    if c in (COL_PROG1, COL_TTP1, "Type of 1st Progression"):
        TIER[c] = "L4"; RATIONALE[c] = "1st-prog landmark — known by definition for eligible cohort"
        continue
    TIER[c] = "L4"; RATIONALE[c] = "Baseline / demographic / molecular"

manifest = pd.DataFrame({"column": list(TIER), "tier": list(TIER.values()),
                         "rationale": [RATIONALE[c] for c in TIER]})
manifest.to_csv(PROC / "leakage_manifest.csv", index=False)
print(f"  wrote {PROC/'leakage_manifest.csv'}  ({len(manifest)} columns)")
print(f"  tier counts: {manifest['tier'].value_counts().to_dict()}")


## 2. Two-level masking on L3 columns

**(a) Row-level mask**: if a treatment STARTED at or after the landmark
day, every cell of (start, end, name, dose, cycles) for that patient is
set to NaN — the model must not see this future treatment.

**(b) Cell-level mask** (NEW): even when a treatment started PRE-landmark,
its END date may be POST-landmark (e.g. ongoing infusion).  We can't
honestly tell the model that the treatment ended on day X if X is in
the future.  We therefore mask any END-day cell whose value > landmark.


In [ ]:
clean = elig.copy()
# (start_col, [comp_cols], [end_cols], cycle_len_col, n_cycles_col)
L3_TIMING = [
    ("Initial Chemo Therapy",
     " Number of days from Diagnosis to Initial Chemo Therapy Start date",
     [" Number of days from Diagnosis to Initial Chemo Therapy end date","Name of Initial Chemo Therapy"],
     [" Number of days from Diagnosis to Initial Chemo Therapy end date"],
     None, None),
    ("Radiation Therapy",
     "Number of days from Diagnosis to Radiation Therapy Start date",
     ["Number of days from Diagnosis to Radiation Therapy end date","Dose","Number of Fractions"],
     ["Number of days from Diagnosis to Radiation Therapy end date"],
     None, None),
    ("Additional Therapy",
     "Number of Days from Diagnosis to Starting Additional Therapy ",
     ["Cycle length of Additional Therapy (q days)",
      "Number of Days from Diagnosis to Complete Additional Therapy ",
      "Number of Cycles of Additional Therapy"],
     ["Number of Days from Diagnosis to Complete Additional Therapy "],
     "Cycle length of Additional Therapy (q days)",
     "Number of Cycles of Additional Therapy"),
    ("Immuno therapy",
     "Number of Days from Diagnosis to Start Immunotherapy ",
     ["Cycle length of Immunotherapy (q days)",
      "Number of Days from Diagnosis to Complete Immunotherapy ",
      "Number of Cycles of Immunotherapy"],
     ["Number of Days from Diagnosis to Complete Immunotherapy "],
     "Cycle length of Immunotherapy (q days)",
     "Number of Cycles of Immunotherapy"),
    ("Brachy therapy",
     "Number of Days from Diagnosis to the day of Insertion of Brachytherapy ",
     [], [], None, None),
    ("Other Types of Therapy (LITT, more chemo, proton therapy)",
     "Number of Days from Diagnosis to Start Other Additional Therapy ",
     ["Number of Days from Diagnosis to Complete Other Additional Therapy "],
     ["Number of Days from Diagnosis to Complete Other Additional Therapy "],
     None, None),
]
n_row_masked  = 0
n_cell_masked = 0
n_cycle_capped = 0
for flag, start_col, comps, end_cols, clen_col, ncyc_col in L3_TIMING:
    if TIER.get(flag) != "L3" or start_col not in clean.columns: continue

    # (a) row-level: drop the whole record if start ≥ landmark
    starts = pd.to_numeric(clean[start_col], errors="coerce")
    row_bad = (starts >= clean["Landmark_day"]) & starts.notna()
    n_row_masked += int(row_bad.sum())
    for c in [flag, start_col, *comps]:
        if c in clean.columns:
            clean.loc[row_bad, c] = np.nan

    # (b) cell-level: end day > landmark even if start was ok
    for ec in end_cols:
        if ec not in clean.columns: continue
        end_vals = pd.to_numeric(clean[ec], errors="coerce")
        cell_bad = (end_vals > clean["Landmark_day"]) & end_vals.notna() & ~row_bad
        n_cell_masked += int(cell_bad.sum())
        clean.loc[cell_bad, ec] = np.nan

    # (c) cycle-count cap: if end-day was masked OR the recorded cycle count
    #     implies a completion day after the landmark, cap n_cycles to whatever
    #     could realistically have been completed by Landmark_day.
    if clen_col and ncyc_col and clen_col in clean.columns and ncyc_col in clean.columns:
        s  = pd.to_numeric(clean[start_col], errors="coerce")
        cl = pd.to_numeric(clean[clen_col],  errors="coerce")
        n  = pd.to_numeric(clean[ncyc_col],  errors="coerce")
        L  = clean["Landmark_day"]
        # Maximum cycles that fit between [start, landmark]
        max_n = ((L - s) / cl).where(cl > 0)
        # Floor (np.floor preserves NaN)
        max_n_floor = np.floor(max_n)
        # Need cap when recorded n exceeds what fits, AND start known
        need_cap = n.notna() & s.notna() & cl.notna() & (n > max_n_floor)
        if need_cap.any():
            capped = max_n_floor.clip(lower=0).astype("Int64")
            n_cycle_capped += int(need_cap.sum())
            clean.loc[need_cap, ncyc_col] = capped[need_cap].astype(float)

print(f"  row-level mask   (start ≥ landmark)         : {n_row_masked} (pt × Tx) records")
print(f"  cell-level mask  (end   >  landmark)        : {n_cell_masked} (pt × end-date) cells")
print(f"  cycle-count cap  (n_cycles → max@landmark)  : {n_cycle_capped} (pt × Tx) cells")


## 3. Drop L1 + L2 columns


In [ ]:
drop_cols = [c for c, t in TIER.items() if t in ("L1","L2")]
clean = clean.drop(columns=drop_cols, errors="ignore")
print(f"  dropped {len(drop_cols)} columns (L1+L2)")
print(f"  clean shape : {clean.shape}")


## 4. Save the artefacts


In [ ]:
clean.to_csv(PROC / "clean_clinical.csv", index=False)
print(f"  wrote {PROC/'clean_clinical.csv'}  ({len(clean)} rows × {clean.shape[1]} cols)")


## 5. Build feature_groups.json — column lists per experiment


In [ ]:
DEMO_DIAG = ["Sex at Birth","Race","Age at diagnosis","Primary Diagnosis",
             "Grade of Primary Brain Tumor","Stereotactic Biopsy before Surgical Resection",
             "Previous Brain Tumor","Type of previous brain tumor",
             "Year of previous surgery","Grade of Previous brain tumor",
             "Number of days from Diagnosis to First surgery or procedure ",
             "Number of days from Diagnosis to date of First Progression",
             "Type of 1st Progression"]

MOLECULAR = ["IDH1 mutation","IDH2 mutation","1p/19q","ATRX mutation","MGMT methylation",
             "BRAF V600E mutation","TERT promoter mutation",
             "Chromosome 7 gain and Chromosome 10 loss","H3-3A mutation",
             "EGFR amplification","PTEN mutation","CDKN2A/B deletion","TP53 alteration",
             "Other mutations/alterations"]

INITIAL_TX = [c for c in [
    "Initial Chemo Therapy","Name of Initial Chemo Therapy",
    " Number of days from Diagnosis to Initial Chemo Therapy Start date",
    " Number of days from Diagnosis to Initial Chemo Therapy end date",
    "Radiation Therapy","Number of days from Diagnosis to Radiation Therapy Start date",
    "Number of days from Diagnosis to Radiation Therapy end date","Dose","Number of Fractions",
] if c in clean.columns]

SALVAGE_TX = [c for c in [
    "Additional Therapy","Cycle length of Additional Therapy (q days)",
    "Number of Days from Diagnosis to Starting Additional Therapy ",
    "Number of Days from Diagnosis to Complete Additional Therapy ",
    "Number of Cycles of Additional Therapy",
    "Immuno therapy","Cycle length of Immunotherapy (q days)",
    "Number of Days from Diagnosis to Start Immunotherapy ",
    "Number of Days from Diagnosis to Complete Immunotherapy ",
    "Number of Cycles of Immunotherapy",
    "Brachy therapy","Number of Days from Diagnosis to the day of Insertion of Brachytherapy ",
    "Other Types of Therapy (LITT, more chemo, proton therapy)",
    "Number of Days from Diagnosis to Start Other Additional Therapy ",
    "Number of Days from Diagnosis to Complete Other Additional Therapy ",
] if c in clean.columns]

# Experiment definitions — strict reading of the supervisor's spec:
#   (i)   Molecular only
#   (ii)  Molecular + Treatment   (initial + salvage gated by T₂)
#   (iii) Molecular + Treatment + Timepoints + Radiomic
#   (iv)  Molecular + Treatment + Timepoints + VLM
# Demographics + diagnosis are stored as a separate 'demo_diag' group so the
# prompt-builder can choose to prepend them uniformly (off by default to
# stay faithful to the spec).
groups = {
    "demo_diag":                 DEMO_DIAG,
    "exp1_molecular":            MOLECULAR,
    "exp2_molecular_treatment":  MOLECULAR + INITIAL_TX + SALVAGE_TX,
    "exp3_clinical_radiomic":    MOLECULAR + INITIAL_TX + SALVAGE_TX,
    "exp4_clinical_vlm":         MOLECULAR + INITIAL_TX + SALVAGE_TX,
}
# Note: exp3 and exp4 add radiomic / VLM features at the prompt-builder stage,
# not inside clean_clinical.csv. Their *clinical* column lists are identical
# to exp2.

# Drop columns no longer in clean (e.g. T2-tier dropped)
for k, lst in groups.items():
    groups[k] = [c for c in lst if c in clean.columns]
    print(f"  {k:32s}  →  {len(groups[k])} clinical cols")

with open(PROC / "feature_groups.json", "w") as f:
    json.dump(groups, f, indent=2)
print(f"\n  wrote {PROC/'feature_groups.json'}")


## 6. Splits — frozen patient assignment

If `splits/split_assignments.csv` exists, reuse it so Second_Recur is
not affected by any later First_Recur re-split. Older checkouts fall
back to the First_Recur split assignment and then freeze the subset.


In [ ]:
from _landmark import first_recur_split_assignments

assign = first_recur_split_assignments()
elig_pids = set(clean["Patient_ID"])
out_rows = {"Train": [], "Validation": [], "Test": []}
for split, pids in assign.items():
    sub = clean[clean["Patient_ID"].isin(pids)]
    out_rows[split] = sub
    n_pos = int((sub["y"]==1).sum()); n_neg = int((sub["y"]==0).sum())
    print(f"  {split:11s}: n={len(sub):3d}  (pos={n_pos}  neg={n_neg})")
    sub.to_csv(SPLITS / f"{split}.csv", index=False)
pd.concat([
    pd.DataFrame({"Patient_ID": out_rows["Train"]["Patient_ID"], "split": "Train"}),
    pd.DataFrame({"Patient_ID": out_rows["Validation"]["Patient_ID"], "split": "Validation"}),
    pd.DataFrame({"Patient_ID": out_rows["Test"]["Patient_ID"], "split": "Test"}),
], ignore_index=True).sort_values("Patient_ID").to_csv(SPLITS / "split_assignments.csv", index=False)
print(f"\n  wrote splits/Train.csv  Validation.csv  Test.csv")
print(f"  wrote {SPLITS/'split_assignments.csv'}")


## 7. Summary report


In [ ]:
summary = {
    "task":                   "Second Progression",
    "raw_patients":           int(len(raw)),
    "eligible_patients":      int(len(elig)),
    "y1_count":               int((elig['y']==1).sum()),
    "y0_count":               int((elig['y']==0).sum()),
    "Landmark_day_median":    float(elig['Landmark_day'].median()),
    "n_columns_dropped_L1":   int(sum(1 for t in TIER.values() if t=="L1")),
    "n_columns_dropped_L2":   int(sum(1 for t in TIER.values() if t=="L2")),
    "n_columns_kept_L3":      int(sum(1 for t in TIER.values() if t=="L3")),
    "n_columns_kept_L4":      int(sum(1 for t in TIER.values() if t=="L4")),
    "splits": {s: {"n": len(out_rows[s]),
                   "pos": int((out_rows[s]['y']==1).sum()),
                   "neg": int((out_rows[s]['y']==0).sum())}
                for s in ("Train","Validation","Test")},
    "experiments_clinical_col_counts": {k: len(v) for k, v in groups.items()},
}
with open(PROC / "preprocessing_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
